你可以先把整个 PyTorch 看成下面这样：

torch
├── nn
├── autograd
├── optim
├── utils
├── onnx
├── cuda
└── ...

其中目前最重要的 4 个：

torch.nn
→ 搭建神经网络

torch.autograd
→ 自动求导

torch.optim
→ 优化参数

torch.utils.data
→ 读取和组织数据

torch：Tensor 与基础计算
torch.nn：神经网络层
torch.nn.functional：常用函数式操作
torch.autograd：自动求导
torch.optim：优化器
torch.utils.data：Dataset / DataLoader

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
print("your torch library in here:{}".format(torch.__path__))
print("your nn library in here:{}".format(nn.__path__))
print("your optim library in here:{}".format(optim.__path__))

your torch library in here:['/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch']
your nn library in here:['/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/nn']
your optim library in here:['/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/optim']


## 模型的训练：
1. 数据 Data
      ↓
2. 模型 Model
      ↓
3. 损失函数 + 优化器
      ↓
4. 训练循环

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms


def main():

    # ============================================================
    # step 1/4 : 数据模块
    # ============================================================

    data_dir = "/home/yyy/desktop/github/PyTorch-Tutorial-2nd/data"

    # MNIST 原图大小：28 × 28
    # 为了适配原教程 TinyCNN，缩放到 8 × 8
    transform = transforms.Compose([
        transforms.Resize((8, 8)),
        transforms.ToTensor(),
    ])

    # 训练集
    train_data = datasets.MNIST(
        root=data_dir,
        train=True,
        transform=transform,
        download=True
    )

    # 测试集
    valid_data = datasets.MNIST(
        root=data_dir,
        train=False,
        transform=transform,
        download=True
    )

    train_loader = DataLoader(
        dataset=train_data,
        batch_size=64,
        shuffle=True
    )

    valid_loader = DataLoader(
        dataset=valid_data,
        batch_size=64,
        shuffle=False
    )

    print("训练集大小:", len(train_data))
    print("验证集大小:", len(valid_data))

    # ============================================================
    # step 2/4 : 模型模块
    # ============================================================

    class TinyCNN(nn.Module):

        def __init__(self, cls_num=10):
            super().__init__()

            # 输入：
            # [batch, 1, 8, 8]
            #
            # 经过 3×3 卷积：
            # [batch, 1, 6, 6]

            self.convolution_layer = nn.Conv2d(
                in_channels=1,
                out_channels=1,
                kernel_size=3
            )

            # 6 × 6 = 36
            self.fc = nn.Linear(36, cls_num)

        def forward(self, x):

            x = self.convolution_layer(x)

            # [batch, 1, 6, 6]
            #       ↓
            # [batch, 36]

            x = x.view(x.size(0), -1)

            out = self.fc(x)

            return out

    # MNIST 有 10 个类别：0~9
    model = TinyCNN(cls_num=10)

    # ============================================================
    # step 3/4 : 优化模块
    # ============================================================

    # 多分类交叉熵
    loss_f = nn.CrossEntropyLoss()

    # SGD 优化器
    optimizer = optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4
    )

    # 每 5 个 epoch 学习率 × 0.1
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=5,
        gamma=0.1
    )

    # ============================================================
    # step 4/4 : 训练模块
    # ============================================================

    for epoch in range(10):

        # --------------------------------------------------------
        # 训练
        # --------------------------------------------------------

        model.train()

        train_correct = 0
        train_total = 0
        train_loss = 0

        for data, labels in train_loader:

            # 1. 前向传播
            outputs = model(data)

            # 2. 计算 loss
            loss = loss_f(outputs, labels)

            # 3. 清空上一轮梯度
            optimizer.zero_grad()

            # 4. 反向传播
            loss.backward()

            # 5. 更新参数
            optimizer.step()

            # 累计 loss
            train_loss += loss.item()

            # 找到预测类别
            _, predicted = torch.max(outputs, dim=1)

            train_total += labels.size(0)

            train_correct += (
                predicted == labels
            ).sum().item()

        train_acc = train_correct / train_total

        # --------------------------------------------------------
        # 验证
        # --------------------------------------------------------

        model.eval()

        valid_correct = 0
        valid_total = 0
        valid_loss = 0

        # 验证时不需要计算梯度
        with torch.no_grad():

            for data, labels in valid_loader:

                # 前向传播
                outputs = model(data)

                # loss
                loss = loss_f(outputs, labels)

                valid_loss += loss.item()

                # 预测
                _, predicted = torch.max(
                    outputs,
                    dim=1
                )

                valid_total += labels.size(0)

                valid_correct += (
                    predicted == labels
                ).sum().item()

        valid_acc = valid_correct / valid_total

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss / len(train_loader):.4f} | "
            f"Train Acc: {train_acc:.2%} | "
            f"Valid Loss: {valid_loss / len(valid_loader):.4f} | "
            f"Valid Acc: {valid_acc:.2%}"
        )

        # 更新学习率
        scheduler.step()


if __name__ == "__main__":
    main()

训练集大小: 60000
验证集大小: 10000
Epoch 01 | Train Loss: 0.5020 | Train Acc: 84.34% | Valid Loss: 0.3798 | Valid Acc: 89.12%
Epoch 02 | Train Loss: 0.4020 | Train Acc: 87.78% | Valid Loss: 0.3780 | Valid Acc: 88.35%
Epoch 03 | Train Loss: 0.3955 | Train Acc: 88.14% | Valid Loss: 0.3946 | Valid Acc: 88.76%
Epoch 04 | Train Loss: 0.3962 | Train Acc: 88.06% | Valid Loss: 0.3751 | Valid Acc: 88.71%
Epoch 05 | Train Loss: 0.3943 | Train Acc: 88.25% | Valid Loss: 0.3768 | Valid Acc: 89.04%
Epoch 06 | Train Loss: 0.3632 | Train Acc: 89.26% | Valid Loss: 0.3402 | Valid Acc: 90.33%
Epoch 07 | Train Loss: 0.3600 | Train Acc: 89.37% | Valid Loss: 0.3377 | Valid Acc: 90.38%
Epoch 08 | Train Loss: 0.3588 | Train Acc: 89.44% | Valid Loss: 0.3380 | Valid Acc: 90.30%
Epoch 09 | Train Loss: 0.3582 | Train Acc: 89.41% | Valid Loss: 0.3385 | Valid Acc: 90.31%
Epoch 10 | Train Loss: 0.3584 | Train Acc: 89.35% | Valid Loss: 0.3382 | Valid Acc: 90.24%


In [4]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image


def main():

    # ============================================================
    # step 1/4 : 数据模块
    # ============================================================

    class COVID19Dataset(Dataset):

        def __init__(self, root_dir, txt_path, transform=None):
            """
            root_dir:
                图片所在目录

            txt_path:
                标签 txt 文件路径

            transform:
                图像预处理方法
            """

            self.root_dir = root_dir
            self.txt_path = txt_path
            self.transform = transform

            # [(image_path, label), ...]
            self.img_info = []

            self._get_img_info()

        def __getitem__(self, index):
            """
            根据 index 读取一张图片及其标签
            """

            path_img, label = self.img_info[index]

            # X 光图像转为灰度图
            img = Image.open(path_img).convert("L")

            if self.transform is not None:
                img = self.transform(img)

            return img, label

        def __len__(self):

            if len(self.img_info) == 0:
                raise Exception(
                    "\ndata_dir:{} is an empty dir! "
                    "Please check your path to images!".format(
                        self.root_dir
                    )
                )

            return len(self.img_info)

        def _get_img_info(self):
            """
            从 txt 中读取：
                图片路径
                标签

            最终保存成：

            [
                (image_path, label),
                ...
            ]
            """

            with open(self.txt_path, "r") as f:

                txt_data = f.read().strip()
                txt_data = txt_data.split("\n")

            self.img_info = [
                (
                    os.path.join(
                        self.root_dir,
                        line.split()[0]
                    ),
                    int(line.split()[2])
                )
                for line in txt_data
            ]

    # ------------------------------------------------------------
    # 数据路径
    # ------------------------------------------------------------

    root_dir = (
        "/home/yyy/desktop/github/"
        "PyTorch-Tutorial-2nd/data/covid-19-demo"
    )

    img_dir = os.path.join(
        root_dir,
        "imgs"
    )

    path_txt_train = os.path.join(
        root_dir,
        "labels",
        "train.txt"
    )

    path_txt_valid = os.path.join(
        root_dir,
        "labels",
        "valid.txt"
    )

    # ------------------------------------------------------------
    # 图像预处理
    # ------------------------------------------------------------

    transforms_func = transforms.Compose([

        # 原图缩放为 8×8
        transforms.Resize((8, 8)),

        # PIL Image → Tensor
        transforms.ToTensor(),
    ])

    # ------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------

    train_data = COVID19Dataset(
        root_dir=img_dir,
        txt_path=path_txt_train,
        transform=transforms_func
    )

    valid_data = COVID19Dataset(
        root_dir=img_dir,
        txt_path=path_txt_valid,
        transform=transforms_func
    )

    # ------------------------------------------------------------
    # DataLoader
    # ------------------------------------------------------------

    train_loader = DataLoader(
        dataset=train_data,
        batch_size=2,
        shuffle=True
    )

    valid_loader = DataLoader(
        dataset=valid_data,
        batch_size=2,
        shuffle=False
    )

    print("train dataset size:", len(train_data))
    print("valid dataset size:", len(valid_data))

    # ============================================================
    # step 2/4 : 模型模块
    # ============================================================

    class TinyCNN(nn.Module):

        def __init__(self, cls_num=2):

            super().__init__()

            # 输入：
            # [batch, 1, 8, 8]
            #
            # Conv2d：
            # kernel = 3×3
            #
            # 输出：
            # [batch, 1, 6, 6]

            self.convolution_layer = nn.Conv2d(
                in_channels=1,
                out_channels=1,
                kernel_size=(3, 3)
            )

            # 1 × 6 × 6 = 36
            self.fc = nn.Linear(
                36,
                cls_num
            )

        def forward(self, x):

            x = self.convolution_layer(x)

            # [batch, 1, 6, 6]
            # →
            # [batch, 36]

            x = x.view(
                x.size(0),
                -1
            )

            out = self.fc(x)

            return out

    model = TinyCNN(
        cls_num=2
    )

    # ============================================================
    # step 3/4 : 优化模块
    # ============================================================

    # 二分类，但这里输出两个类别 logits
    # 所以使用 CrossEntropyLoss
    loss_f = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        gamma=0.1,
        step_size=50
    )

    # ============================================================
    # step 4/4 : 训练模块
    # ============================================================

    for epoch in range(100):

        # --------------------------------------------------------
        # 训练
        # --------------------------------------------------------

        model.train()

        for data, labels in train_loader:

            # 1. 前向传播
            outputs = model(data)

            # 2. 清空上一轮梯度
            optimizer.zero_grad()

            # 3. 计算 loss
            loss = loss_f(
                outputs,
                labels
            )

            # 4. 反向传播
            loss.backward()

            # 5. 更新参数
            optimizer.step()

            # ----------------------------------------------------
            # 计算分类准确率
            # ----------------------------------------------------

            _, predicted = torch.max(
                outputs,
                dim=1
            )

            correct_num = (
                predicted == labels
            ).sum()

            acc = (
                correct_num
                / labels.shape[0]
            )

            print(
                "Epoch:{} "
                "Train Loss:{:.4f} "
                "Acc:{:.0%}".format(
                    epoch,
                    loss.item(),
                    acc.item()
                )
            )

        # --------------------------------------------------------
        # 验证
        # --------------------------------------------------------

        model.eval()

        with torch.no_grad():

            for data, labels in valid_loader:

                # forward
                outputs = model(data)

                # loss
                loss = loss_f(
                    outputs,
                    labels
                )

                # 分类结果
                _, predicted = torch.max(
                    outputs,
                    dim=1
                )

                correct_num = (
                    predicted == labels
                ).sum()

                acc_valid = (
                    correct_num
                    / labels.shape[0]
                )

                print(
                    "Epoch:{} "
                    "Valid Loss:{:.4f} "
                    "Acc:{:.0%}".format(
                        epoch,
                        loss.item(),
                        acc_valid.item()
                    )
                )

        # --------------------------------------------------------
        # 停止条件
        # --------------------------------------------------------

        if acc_valid == 1:
            break

        # --------------------------------------------------------
        # 学习率调整
        # --------------------------------------------------------

        scheduler.step()


if __name__ == "__main__":
    main()

train dataset size: 2
valid dataset size: 2
Epoch:0 Train Loss:0.6962 Acc:50%
Epoch:0 Valid Loss:0.6782 Acc:100%


## 2.3 核心数据结构——Tensor

Tensor
├── 数据本身
├── 数据类型
├── 形状
├── 所在设备
└── 自动求导信息

第一组：描述数据
data
dtype
shape
device
第二组：描述自动求导
requires_grad
grad
grad_fn
is_leaf

In [2]:
import torch

x = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True
)

print(x.shape)
x = torch.randn(32, 3, 224, 224)
print(x.shape)
a = torch.tensor([1, 2, 3])
b = torch.tensor([1., 2., 3.])

print(a.dtype)
print(b.dtype)

torch.Size([3])
torch.Size([32, 3, 224, 224])
torch.int64
torch.float32


深度学习中最常见的是：

torch.float32

另外以后你还会大量遇到：

float16
bfloat16
int64

尤其：

模型权重/激活
→ float32 / float16 / bfloat16

分类标签
→ 常见 int64

例如 CrossEntropyLoss 的类别标签通常就是 torch.long，也就是 int64。

In [ ]:
x = torch.tensor([1., 2.])

print(x.device)

## 2.4 张量的相关函数

接下来开始学习各类张量的api，主要参考官方文档，通过右边目录栏可以看出有以下几个部分。

torchTensors
Generators
Random sampling
Serialization
Parallelism
Locally disabling gradient computation
Math operations
Utilities

torch.tensor(data, dtype=None, device=None, requires_grad=False, pin_memory=False)

data(array_like) - tensor的初始数据，可以是list, tuple, numpy array, scalar或其他类型。

dtype(torch.dtype, optional) - tensor的数据类型，如torch.uint8, torch.float, torch.long等

device (torch.device, optional) – 决定tensor位于cpu还是gpu。如果为None，将会采用默认值，默认值在torch.set_default_tensor_type()中设置，默认为 cpu。

requires_grad (bool, optional) – 决定是否需要计算梯度。

pin_memory (bool, optional) – 是否将tensor存于锁页内存。这与内存的存储方式有关，通常为False。

In [ ]:
import torch
import numpy as np

l = [[1., -1.], [1., -1.]]
t_from_list = torch.tensor(l)
arr = np.array([[1, 2, 3], [4, 5, 6]])
t_from_array = torch.tensor(arr)

print(t_from_list, t_from_list.dtype)
print(t_from_array, t_from_array.dtype)

tensor([[ 1., -1.],
        [ 1., -1.]]) torch.float32
tensor([[1, 2, 3],
        [4, 5, 6]]) torch.int64
